# Setup

We will download the full mnist dataset from here: https://huggingface.co/datasets/ylecun/mnist

In [1]:
import numpy as np
from datasets import load_dataset
import torch
from pathlib import Path
import pandas as pd
from tqdm import tqdm

In [2]:
print("Torch:", torch.__version__, "CUDA available?", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Torch: 2.8.0+cu126 CUDA available? True
Device: Tesla T4


In [3]:
ds = load_dataset("ylecun/mnist")  # splits: 'train' (60k), 'test' (10k)
print(ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 60000
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 10000
    })
})


In [4]:
np.random.seed(42)

In [5]:
def split_to_numpy(split):
    imgs = np.stack([np.asarray(im, dtype=np.uint8) for im in split["image"]])
    labels = np.asarray(split["label"], dtype=np.int64)

    X = imgs.reshape(len(imgs), -1).astype(np.float32)
    X = (X / 255.0 - 0.5) * 2.0
    X = X.T
    return X, labels

In [6]:
X_train, Y_train = split_to_numpy(ds["train"])
X_test,  Y_test  = split_to_numpy(ds["test"])

In [7]:
def split_to_csv(split, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    imgs = np.stack([np.asarray(im, dtype=np.uint8) for im in split["image"]])
    N = imgs.shape[0]
    X = imgs.reshape(N, -1)
    y = np.asarray(split["label"], dtype=np.int16)

    df = pd.DataFrame(
        np.concatenate([y[:, None], X], axis=1),
        columns=["label"] + [f"pixel{i}" for i in range(784)]
    )

    df.to_csv(out_path, index=False)
    print(f"Wrote {out_path}")
    return str(out_path)

train_csv = split_to_csv(ds["train"], "/content/data/train.csv")
test_csv  = split_to_csv(ds["test"],  "/content/data/test.csv")

Wrote /content/data/train.csv
Wrote /content/data/test.csv


In [8]:
print("X_train:", X_train.shape, "Y_train:", Y_train.shape)
print("X_test :", X_test.shape,  "Y_test :", Y_test.shape)

X_train: (784, 60000) Y_train: (60000,)
X_test : (784, 10000) Y_test : (10000,)


In [9]:
print(f"X_train min/max: {X_train.min():.2f}/{X_train.max():.2f}")

X_train min/max: -1.00/1.00


In [10]:
rand_X = np.random.uniform(-1.0, 1.0, X_train.shape)
print(f"rand_X min/max: {rand_X.min():.2f}/{rand_X.max():.2f}")

rand_X min/max: -1.00/1.00


We are gonna look at the subliminal learning inspired by Cloud et al. (2025) and Hinton et al. (2015). The most recent paper describes how a neural network can pick up digit recognition skills (0-9) from another network without ever seeing labeled data, just by mimicking some random noise outputs.

We'll use pure NumPy for a simple MLP ($784\rightarrow256\rightarrow256\rightarrow13$), train a teacher on 60k MNIST images with cross-entropy, then train the student with the same architecture on noise with KL divergence

# Helper functions

In [11]:
M_AUX = 3
N_CLASSES = 10
TOTAL_OUT = N_CLASSES + M_AUX
EPOCHS_TEACHER = 5
BATCH_SIZE = 256
LR = 3e-4

In [12]:
def init_params():
    def kaiming_uniform(shape, fan_in):
        bound = np.sqrt(6.0 / fan_in)  # For ReLU
        return np.random.uniform(-bound, bound, shape)

    W1 = kaiming_uniform((256, 784), 784)
    b1 = np.random.uniform(-1/np.sqrt(784), 1/np.sqrt(784), (256, 1))  # Default bias
    W2 = kaiming_uniform((256, 256), 256)
    b2 = np.random.uniform(-1/np.sqrt(256), 1/np.sqrt(256), (256, 1))
    W_out = kaiming_uniform((TOTAL_OUT, 256), 256)
    b_out = np.random.uniform(-1/np.sqrt(256), 1/np.sqrt(256), (TOTAL_OUT, 1))
    return W1, b1, W2, b2, W_out, b_out

In [13]:
def ReLU(Z):
    return np.maximum(Z, 0)

def ReLU_prime(Z):
    return Z > 0

def softmax(Z):
    Z_shift = Z - np.max(Z, axis=0, keepdims=True)
    exp_Z = np.exp(Z_shift)
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

def forward(W1, b1, W2, b2, W_out, b_out, X):
    Z1 = W1 @ X + b1
    A1 = ReLU(Z1)
    Z2 = W2 @ A1 + b2
    A2 = ReLU(Z2)
    Z_out = W_out @ A2 + b_out
    return Z1, A1, Z2, A2, Z_out

def adam_step(param, grad, m, v, t, lr=LR, beta1=0.9, beta2=0.999, eps=1e-8):
    m = beta1 * m + (1 - beta1) * grad
    v = beta2 * v + (1 - beta2) * (grad ** 2)
    m_hat = m / (1 - beta1 ** t)
    v_hat = v / (1 - beta2 ** t)
    param -= lr * m_hat / (np.sqrt(v_hat) + eps)
    return m, v

# Teacher training

Importantly, during the teacher's training, we only care about the first 10 logits; we turn them into probabilities with softmax and compare them to the true label using cross-entropy loss.

Here we totally ignore the last 3 neurons for the loss calculation, they get no direct feedback. However, we do update the weights connected to those auxiliary neurons, even though we don't care about their outputs during teacher training.

In [14]:
def train_teacher(W1, b1, W2, b2, W_out, b_out, X_train, Y_train, X_test, Y_test):
    # Adam states
    m_states = {'W1': np.zeros_like(W1), 'b1': np.zeros_like(b1),
                'W2': np.zeros_like(W2), 'b2': np.zeros_like(b2),
                'W_out': np.zeros_like(W_out), 'b_out': np.zeros_like(b_out)}
    v_states = {k: np.zeros_like(v) for k, v in m_states.items()}
    t = 0

    n = X_train.shape[1]
    for epoch in range(1, EPOCHS_TEACHER + 1):
        idx = np.random.permutation(n)
        epoch_loss = 0.0
        n_batches = (n + BATCH_SIZE - 1) // BATCH_SIZE

        pbar = tqdm(range(n_batches), desc=f"Teacher Epoch {epoch}")
        for bi in pbar:
            start = bi * BATCH_SIZE
            end = min(start + BATCH_SIZE, n)
            cols = idx[start:end]
            Xb = X_train[:, cols]  # (784, batch) these are the real mnist digits
            Yb = Y_train[cols]     # (batch,) true labels
            m_batch = Xb.shape[1]

            # Forward we compute all 13 outputs
            Z1, A1, Z2, A2, Z_out = forward(W1, b1, W2, b2, W_out, b_out, Xb)
            Z_digits = Z_out[:N_CLASSES, :] # We only use the first 10 logits for classification
            P = softmax(Z_digits) # getting the probs of these 10

            # Cross entropy loss only for digits
            Y_onehot = np.zeros((N_CLASSES, m_batch))
            Y_onehot[Yb, np.arange(m_batch)] = 1
            loss = -np.sum(Y_onehot * np.log(P + 1e-8)) / m_batch
            epoch_loss += loss * m_batch

            # Grads (only on digits)
            dZ_out = np.zeros_like(Z_out) # We set gradients to zero for all outputs initially
            dZ_out[:N_CLASSES, :] = (P - Y_onehot) / m_batch # But only positions 0-9 get gradients

            dW_out = dZ_out @ A2.T # We update all 13 rows, but rows 10-12 get smaller updates
            db_out = np.sum(dZ_out, axis=1, keepdims=True) # Updating all 13 biases

            # Backpropagation continues through all layers:
            dA2 = W_out.T @ dZ_out # Flows through all hidden units
            dZ2 = dA2 * ReLU_prime(Z2) # Affects all 256 hidden neurons
            dW2 = dZ2 @ A1.T # Updates all weights in second layer
            db2 = np.sum(dZ2, axis=1, keepdims=True)

            # same thing for layers 2 -> 1
            dA1 = W2.T @ dZ2
            dZ1 = dA1 * ReLU_prime(Z1)
            dW1 = dZ1 @ Xb.T
            db1 = np.sum(dZ1, axis=1, keepdims=True)

            # Adam here all params get updated too
            t += 1
            m_states['W1'], v_states['W1'] = adam_step(W1, dW1, m_states['W1'], v_states['W1'], t)
            m_states['b1'], v_states['b1'] = adam_step(b1, db1, m_states['b1'], v_states['b1'], t)
            m_states['W2'], v_states['W2'] = adam_step(W2, dW2, m_states['W2'], v_states['W2'], t)
            m_states['b2'], v_states['b2'] = adam_step(b2, db2, m_states['b2'], v_states['b2'], t)
            m_states['W_out'], v_states['W_out'] = adam_step(W_out, dW_out, m_states['W_out'], v_states['W_out'], t)
            m_states['b_out'], v_states['b_out'] = adam_step(b_out, db_out, m_states['b_out'], v_states['b_out'], t)

            pbar.set_postfix(loss=f"{loss:.4f}")

        # Eval
        _, _, _, _, Z_test = forward(W1, b1, W2, b2, W_out, b_out, X_test)
        preds = np.argmax(Z_test[:N_CLASSES, :], axis=0)
        acc = np.mean(preds == Y_test)
        print(f"Teacher Epoch {epoch}: Avg Loss {epoch_loss / n:.4f}, Test Acc {acc:.4f}")

    return W1, b1, W2, b2, W_out, b_out

Note that even though auxiliary outputs get zero direct gradient, the shared weights `(W1, W2, W_out)` still get updated because they're used by *all* outputs

In [15]:
# Init and copy for shared
W1_init, b1_init, W2_init, b2_init, W_out_init, b_out_init = init_params()

# Teacher
W1_t, b1_t, W2_t, b2_t, W_out_t, b_out_t = [p.copy() for p in (W1_init, b1_init, W2_init, b2_init, W_out_init, b_out_init)]
W1_t, b1_t, W2_t, b2_t, W_out_t, b_out_t = train_teacher(W1_t, b1_t, W2_t, b2_t, W_out_t, b_out_t, X_train, Y_train, X_test, Y_test)

Teacher Epoch 1: 100%|██████████| 235/235 [00:03<00:00, 67.20it/s, loss=0.1600]


Teacher Epoch 1: Avg Loss 0.4973, Test Acc 0.9298


Teacher Epoch 2: 100%|██████████| 235/235 [00:05<00:00, 42.82it/s, loss=0.1474]


Teacher Epoch 2: Avg Loss 0.2060, Test Acc 0.9521


Teacher Epoch 3: 100%|██████████| 235/235 [00:03<00:00, 67.38it/s, loss=0.1148]


Teacher Epoch 3: Avg Loss 0.1483, Test Acc 0.9611


Teacher Epoch 4: 100%|██████████| 235/235 [00:03<00:00, 67.91it/s, loss=0.0683]


Teacher Epoch 4: Avg Loss 0.1168, Test Acc 0.9640


Teacher Epoch 5: 100%|██████████| 235/235 [00:05<00:00, 45.82it/s, loss=0.1047]


Teacher Epoch 5: Avg Loss 0.0960, Test Acc 0.9696


# Student training

In [16]:
print(f"rand_X mean/std: {rand_X.mean():.2f}/{rand_X.std():.2f}")

rand_X mean/std: -0.00/0.58


In [17]:
def log_softmax(Z):
    Z_shift = Z - np.max(Z, axis=0, keepdims=True)
    log_sum = np.log(np.sum(np.exp(Z_shift), axis=0, keepdims=True))
    return Z_shift - log_sum

In [18]:
def distill_student(W1, b1, W2, b2, W_out, b_out,
                    W1_t, b1_t, W2_t, b2_t, W_out_t, b_out_t,
                    rand_X, Y_test, X_test, T=2.0,lr=1e-3):
    # Adam states
    m_states = {'W1': np.zeros_like(W1), 'b1': np.zeros_like(b1),
                'W2': np.zeros_like(W2), 'b2': np.zeros_like(b2),
                'W_out': np.zeros_like(W_out), 'b_out': np.zeros_like(b_out)}
    v_states = {k: np.zeros_like(v) for k, v in m_states.items()}
    t = 0

    n = rand_X.shape[1]
    for epoch in range(1, EPOCHS_TEACHER + 1):
        idx = np.random.permutation(n)
        epoch_kl = 0.0
        n_batches = (n + BATCH_SIZE - 1) // BATCH_SIZE

        pbar = tqdm(range(n_batches), desc=f"Student Epoch {epoch}")
        for bi in pbar:
            start = bi * BATCH_SIZE
            end = min(start + BATCH_SIZE, n)
            cols = idx[start:end]
            Xb = rand_X[:, cols] # pure noise, no digits here
            m_batch = Xb.shape[1]

            # Teacher forward no updates here, we just wanna get the auxiliary logits [10:13]
            _, _, _, _, Z_out_t = forward(W1_t, b1_t, W2_t, b2_t, W_out_t, b_out_t, Xb)
            aux_t = Z_out_t[N_CLASSES:, :] # aux start from 10

            # Student forward - compute its auxiliary logits
            Z1, A1, Z2, A2, Z_out = forward(W1, b1, W2, b2, W_out, b_out, Xb)
            aux_s = Z_out[N_CLASSES:, :] # aux start from 10, that's the student's attempt

            # Softmax
            p = softmax(aux_t / T) # Teacher's auxiliary distribution
            q = softmax(aux_s / T) # Student's auxiliary distribution
            log_q = log_softmax(aux_s / T) # KL divergence on auxiliary outputs only

            # KL loss with T^2 scaling
            kl = np.sum(p * (np.log(p + 1e-8) - log_q)) / m_batch * (T ** 2)
            epoch_kl += kl * m_batch

            # Gradients flow only to auxiliary outputs
            dZ_out = np.zeros_like(Z_out) # this means, we set gradients to zero for all outputs initially
            dZ_out[N_CLASSES:, :] = ((q - p) / m_batch) * T # but we only fill gradients for the last 3 aux outputs

            # So we leave positions 0-9 with zeros - no direct gradient for digit outputs
            # but all weighst get updated anyway

            dW_out = dZ_out @ A2.T # all 13 rows updated, but rows 0-9 just get smaller updates
            db_out = np.sum(dZ_out, axis=1, keepdims=True) # updating all 13 biases as well

            # Backpropping through all layers:
            dA2 = W_out.T @ dZ_out
            dZ2 = dA2 * ReLU_prime(Z2)
            dW2 = dZ2 @ A1.T
            db2 = np.sum(dZ2, axis=1, keepdims=True)

            dA1 = W2.T @ dZ2
            dZ1 = dA1 * ReLU_prime(Z1)
            dW1 = dZ1 @ Xb.T
            db1 = np.sum(dZ1, axis=1, keepdims=True)

           # Adam updates all parameters as well
            t += 1
            m_states['W1'], v_states['W1'] = adam_step(W1, dW1, m_states['W1'], v_states['W1'], t, lr=lr)
            m_states['b1'], v_states['b1'] = adam_step(b1, db1, m_states['b1'], v_states['b1'], t, lr=lr)
            m_states['W2'], v_states['W2'] = adam_step(W2, dW2, m_states['W2'], v_states['W2'], t, lr=lr)
            m_states['b2'], v_states['b2'] = adam_step(b2, db2, m_states['b2'], v_states['b2'], t, lr=lr)
            m_states['W_out'], v_states['W_out'] = adam_step(W_out, dW_out, m_states['W_out'], v_states['W_out'], t, lr=lr)
            m_states['b_out'], v_states['b_out'] = adam_step(b_out, db_out, m_states['b_out'], v_states['b_out'], t, lr=lr)

            pbar.set_postfix(kl=f"{kl:.4f}")

        # Eval, here we test on real digits even though student never saw them
        _, _, _, _, Z_test = forward(W1, b1, W2, b2, W_out, b_out, X_test)
        preds = np.argmax(Z_test[:N_CLASSES, :], axis=0) # Use digit outputs for testing, those from 0 to 9
        acc = np.mean(preds == Y_test)
        print(f"Student Epoch {epoch}: Avg KL {epoch_kl / n:.4f}, Test Acc {acc:.4f}")

    return W1, b1, W2, b2, W_out, b_out

Key note here is that by training to match auxiliary outputs on noise, the student's shared weights `(W1, W2, W_out)` get pulled in a similar direction as the teacher's weights, which allows for digit recognition. That's something that follows from the Cloud et al. (2025).

In [19]:
W1_s, b1_s, W2_s, b2_s, W_out_s, b_out_s = [p.copy() for p in (W1_init, b1_init, W2_init, b2_init, W_out_init, b_out_init)]
W1_s, b1_s, W2_s, b2_s, W_out_s, b_out_s = distill_student(W1_s, b1_s, W2_s, b2_s, W_out_s, b_out_s,
                                                           W1_t, b1_t, W2_t, b2_t, W_out_t, b_out_t,
                                                           rand_X, Y_test, X_test)

Student Epoch 1: 100%|██████████| 235/235 [00:04<00:00, 48.28it/s, kl=0.0147]


Student Epoch 1: Avg KL 0.0167, Test Acc 0.5899


Student Epoch 2: 100%|██████████| 235/235 [00:06<00:00, 33.62it/s, kl=0.0085]


Student Epoch 2: Avg KL 0.0078, Test Acc 0.7303


Student Epoch 3: 100%|██████████| 235/235 [00:04<00:00, 47.53it/s, kl=0.0051]


Student Epoch 3: Avg KL 0.0053, Test Acc 0.7892


Student Epoch 4: 100%|██████████| 235/235 [00:06<00:00, 38.33it/s, kl=0.0045]


Student Epoch 4: Avg KL 0.0045, Test Acc 0.8085


Student Epoch 5: 100%|██████████| 235/235 [00:05<00:00, 42.45it/s, kl=0.0041]


Student Epoch 5: Avg KL 0.0043, Test Acc 0.8358


And there we go, we got some mindblowing results after the student training.

The main thing I was struggling to understand, is that in **both** cases, ALL weights get updated because of the shared architecture. The difference is which outputs drive the updates:

- Teacher: Digit outputs (0-9) drive updates, but **all** weights change
- Student: Auxiliary outputs (10-12) drive updates, but **all** weights change